In [22]:
import jax
jax.config.update('jax_platform_name', 'cpu')

print(jax.devices())

from diffrax import ODETerm
from jax.scipy.stats import norm,poisson
import diffrax
import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt

from time import perf_counter
from algorithms.EKI import EnsembleKalmanInversion



[CpuDevice(id=0)]


In [23]:
beta = jnp.log(0.4)
eta = jnp.log(1 / 7)
gamma = jnp.log(1 / 14)
q = jnp.log(0.1)
mu = jnp.log(0.01)
I0 = jnp.log(0.01)
rho = jnp.log(0.4)

par_true = jnp.array([beta, eta, gamma, q, mu, I0, rho])

t_vec = jnp.linspace(0, 100, 100)


def SEIR(t, y, args):
    beta, eta, gamma, q, mu,_,_ = args

    beta = jnp.exp(beta)
    eta = jnp.exp(eta)
    gamma = jnp.exp(gamma)
    q = jnp.exp(q)
    mu = jnp.exp(mu)

    S, E, I, R = y
    N = S + E + I + R

    dS = -beta * ((E + q * I) / N) * S
    dE = beta * ((E + q * I) / N) * S - eta * E - gamma * E
    dI = eta * E - gamma * I - mu * I
    dR = gamma * E + gamma * I

    return jnp.array([dS, dE, dI, dR])

In [24]:
def model(ts, y0, par):
    solution = diffrax.diffeqsolve(
        ODETerm(SEIR),
        diffrax.Dopri5(),
        t0=ts[0],
        t1=ts[-1],
        dt0=0.1,
        y0=y0,
        args=par,
        saveat=diffrax.SaveAt(ts=ts),
    )
    return solution.ys


rng_key = jax.random.PRNGKey(0)

N = 1000
y_true = model(
    t_vec,
    jnp.array([N - N * jnp.exp(I0), N * jnp.exp(I0), 0, 0]),
    (beta, eta, gamma, q, mu, I0, rho),
)

key = jax.random.key(0)

key, I_noise_key = jax.random.split(key)
data_newI = jax.random.poisson(I_noise_key, jnp.exp(rho) * jnp.exp(eta) * y_true[:, 1] + 0.005)

key, D_noise_key = jax.random.split(key)
data_newD = jax.random.poisson(D_noise_key, jnp.exp(mu) * y_true[:, 2] + 0.005)

In [25]:
observations = jnp.concatenate(
    (data_newI[..., jnp.newaxis], data_newD[..., jnp.newaxis]), axis=-1
).reshape(-1)


In [26]:
def map(par, _):
    full_state = jnp.maximum(model(t_vec, jnp.array([1000, 1.0, 1.0, 0]), par), 1e-6)

    data_newI = jnp.maximum(jnp.exp(eta) * full_state[:, 1], 1e-6)

    data_newD = jnp.maximum(jnp.exp(mu) * full_state[:, 2], 1e-6)

    return jnp.concatenate(
        (data_newI[..., jnp.newaxis], data_newD[..., jnp.newaxis]), axis=-1
    ).reshape(-1)

In [27]:
rng_key, noise_key = jax.random.split(rng_key)

dim_output = len(par_true)
gamma_mat = 10 * jnp.eye(len(observations))

num_ensemble_members = 100

rng_key, init_ensemble_key, init_key = jax.random.split(rng_key, 3)

initial_ensemble = jnp.log(
    jnp.array([0.1, 0.1, 0.1, 0.1, 0.01, 0.01, 0.1])
) + 0.1 * jax.random.normal(init_ensemble_key, (num_ensemble_members, dim_output))

(final_ensemble, _), _ = EnsembleKalmanInversion(
    initial_ensemble,
    observations,
    gamma_mat,
    lambda x, y: jax.vmap(map, in_axes=(0, 0))(x, y),
    0.001,
    init_key,
)

par_estimate = jnp.mean(final_ensemble, axis=0)

print(jnp.exp(par_estimate))
print(f"Relative Error: {jnp.abs(par_true - par_estimate) / par_true}")

[0.1009192  0.09946112 0.09821646 0.1000423  0.01007069 0.00992152
 0.10064633]
Relative Error: [-1.5029557e+00 -1.8607144e-01 -1.2067792e-01 -1.8368683e-04
 -1.5296537e-03 -1.7109589e-03 -1.5059105e+00]
